# Week 1 CSI Data Analysis\n\nThis notebook reads recorded/provided CSI Parquet data, builds an amplitude matrix, plots a heatmap, and computes a simple motion score.

In [ ]:
from pathlib import Path\nimport sys\n\nrepo_root = Path.cwd()\nif not (repo_root / 'server').exists():\n    repo_root = repo_root.parent\nsys.path.insert(0, str(repo_root / 'server'))\n\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom csi_utils import build_amplitude_matrix, compute_motion_score, compute_dynamic_threshold, debounce_motion_decision

## Load Data\n\nPut the provided Parquet file under `data/`. Update the path below if your file name is different.

In [ ]:
data_path = repo_root / 'data' / 'csi_final.parquet'\ndf = pd.read_parquet(data_path)\n\nprint('Rows:', len(df))\nprint('Columns:', list(df.columns))\ndisplay(df.head())

## Build Amplitude Matrix

In [ ]:
amplitude_matrix, source_column = build_amplitude_matrix(df)\nprint('CSI source column:', source_column)\nprint('Amplitude matrix shape:', amplitude_matrix.shape)

## Heatmap\n\nDark horizontal bands may correspond to guard/null subcarriers. These subcarriers are not used for payload data and can appear as low-energy bands.

In [ ]:
frames = amplitude_matrix[:1000]\n\nplt.figure(figsize=(12, 6))\nplt.imshow(frames.T, aspect='auto')\nplt.colorbar(label='Amplitude')\nplt.title('CSI Amplitude Heatmap')\nplt.xlabel('Time / packet index')\nplt.ylabel('Subcarrier index')\nplt.show()

## Motion Score\n\nThis baseline motion score uses rolling variance over CSI amplitude values. It is not a final ML model.

In [ ]:
motion_score = compute_motion_score(amplitude_matrix)\nthreshold = compute_dynamic_threshold(motion_score)\ndecision = debounce_motion_decision(motion_score, threshold)\n\nplt.figure(figsize=(12, 5))\nplt.plot(motion_score.values, label='Motion score')\nplt.axhline(y=threshold, linestyle='--', label=f'Threshold ({threshold:.2f})')\nplt.fill_between(range(len(motion_score)), 0, motion_score.values, where=decision.values, alpha=0.25, label='Motion decision')\nplt.title('CSI Motion Score')\nplt.xlabel('Time / packet index')\nplt.ylabel('Motion score')\nplt.legend()\nplt.grid(True)\nplt.show()